# 3-4. 컨텍스트 압축 (Contextual Compression)

<!-- optional -->

## 학습 목표
- `LLMChainExtractor`로 검색된 문서에서 **질문과 관련된 부분만** 추출해 컨텍스트를 줄일 수 있다.
- Parent Document Retriever(Small-to-Big) 패턴의 필요성을 설명하고 구현할 수 있다.
- Anthropic의 **Contextual Retrieval** 기법 아이디어를 이해한다.

## 핵심 키워드
LLMChainExtractor · ParentDocumentRetriever · Small-to-Big · Contextual Retrieval


In [ ]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '../..')

from common import get_chat_model, get_embeddings, provider_badge
from common.ko_tokenizer import tokenize_ko
from common.loaders import load_any

print(provider_badge())


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

docs = load_any('../../data/corpus_ko.txt')
chunks = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50).split_documents(docs)
vectordb = FAISS.from_documents(chunks, get_embeddings())
base_retriever = vectordb.as_retriever(search_kwargs={'k': 5})
llm = get_chat_model(temperature=0.0)


## 1. LLMChainExtractor: 문서 속 "관련 문장만" 뽑기

청크 크기를 크게 잡아두면 recall은 좋지만 **프롬프트 토큰 비용**이 증가한다. LLM으로 각 문서에서 질문과 관련된 문장만 추출하면 최종 컨텍스트를 10~30%로 줄일 수 있다.


In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

extractor = LLMChainExtractor.from_llm(llm)
compressed_retriever = ContextualCompressionRetriever(
    base_compressor=extractor,
    base_retriever=base_retriever,
)

q = '개인정보보호법에서 개인정보의 정의는?'
original = base_retriever.invoke(q)
compressed = compressed_retriever.invoke(q)

print('=== 원본 ===')
for d in original:
    print(f'[{len(d.page_content)}자] {d.page_content[:80]}...')
print('\n=== 압축 ===')
for d in compressed:
    print(f'[{len(d.page_content)}자] {d.page_content}')


## 2. Parent Document Retriever (Small-to-Big)

딜레마:
- 작은 청크 → 검색 정확도 ↑, 문맥 ↓
- 큰 청크 → 문맥 ↑, 검색 정확도 ↓

해결: **작은 청크로 검색하고, 검색된 작은 청크가 속한 큰 부모 청크를 LLM 컨텍스트로 사용.**


In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
import uuid

# 부모(큰)·자식(작은) 분할기
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)

# 자식 청크 검색용 벡터스토어와 부모 저장용 docstore 준비
# (자식 청크는 add_documents 호출 시 자동 생성·임베딩된다)
child_vectordb = FAISS.from_documents(
    child_splitter.split_documents(docs),
    get_embeddings(),
)
store = InMemoryStore()

parent_retriever = ParentDocumentRetriever(
    vectorstore=child_vectordb,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
parent_retriever.add_documents(docs, ids=[str(uuid.uuid4()) for _ in docs])

q = 'RAG가 환각에 어떻게 도움이 되는가?'
results = parent_retriever.invoke(q)
for d in results:
    print(f'[{len(d.page_content)}자] {d.page_content[:100]}...')


## 3. Contextual Retrieval (Anthropic, 2024)

**문제**: 청크는 원래 문서의 맥락을 잃는다. "이 규정은 ~를 적용한다"의 *이*가 무엇인지 청크만 보면 모른다.

**해결**: 각 청크에 대해 LLM이 **50~100 토큰짜리 맥락 설명**을 생성해 청크 앞에 붙인 뒤 임베딩/BM25 인덱싱한다.

```
원본 청크: "이 규정은 전산실에 적용된다."
 ▼ (LLM으로 맥락 추가)
보강 청크: "[맥락: 전자금융감독규정 제3장 물리보안 섹션의 일부] 이 규정은 전산실에 적용된다."
```

Anthropic 실험에서는 실패 검색율 49% 감소가 보고되었다.
**비용**: 인덱싱 1회당 청크마다 LLM 호출 1회 — prompt caching으로 최대 90% 절감 가능.


In [ ]:
# 간단 구현 (데모용)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

ctx_prompt = ChatPromptTemplate.from_template(
    '다음은 전체 문서이다:\n<doc>{whole}</doc>\n\n'
    '이 문서의 일부 청크:\n<chunk>{chunk}</chunk>\n\n'
    '이 청크가 전체 문서에서 어떤 역할을 하는지 1~2문장의 간결한 맥락 설명만 출력하라.'
)
ctx_chain = ctx_prompt | llm | StrOutputParser()

whole = docs[0].page_content
sample_chunk = chunks[3].page_content
ctx = ctx_chain.invoke({'whole': whole, 'chunk': sample_chunk})
print('원본 청크:', sample_chunk[:120])
print('\n생성된 맥락:', ctx)
print('\n보강 청크:', f'[맥락: {ctx}] {sample_chunk[:100]}')
